In [9]:
!export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/home/dongyoonhwang/.mujoco/mujoco210/bin:/usr/lib/nvidia
!export MUJOCO_PY_MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJLIB_PATH=/home/nas2_userI/byungkunlee/.mujoco/mujoco210/bin/libmujoco210.so
!export MUJOCO_PY_MUJOCO_PATH=/home/dongyoonhwang/.mujoco/mujoco210

# >>> MuJoCo visualization >>>
!export XLA_PYTHON_CLIENT_PREALLOCATE=false
!export MUJOCO_GL='egl'
!export MUJOCO_EGL_DEVICE_ID='0'
!export MKL_SERVICE_FORCE_INTEL='0'

In [1]:
import sys
import os
import math
import pandas as pd
sys.path.append("../")
sys.path.append("../..")

In [2]:
from scale_rl.common.wandb_utils import *

In [3]:
simbav2_rr1_eval_df = pd.read_csv('../../results/simbaV2_utd1.csv', index_col=0)
simbav2_rr2_eval_df = pd.read_csv('../../results/simbaV2_utd2.csv', index_col=0)
simbav2_rr4_eval_df = pd.read_csv('../../results/simbaV2_utd4.csv', index_col=0)
simbav2_rr8_eval_df = pd.read_csv('../../results/simbaV2_utd8.csv', index_col=0)

simbav2_rr1_eval_df["exp_name"] = "UTD = 1"
simbav2_rr2_eval_df["exp_name"] = "UTD = 2"
simbav2_rr4_eval_df["exp_name"] = "UTD = 4"
simbav2_rr8_eval_df["exp_name"] = "UTD = 8"

#### Collection

In [4]:
eval_df = pd.concat([
    simbav2_rr1_eval_df,
    simbav2_rr2_eval_df,
    simbav2_rr4_eval_df,
    simbav2_rr8_eval_df,
    ], ignore_index=True, sort=False)
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
eval_df

,exp_name,env_name,seed,metric,env_step,value
0,UTD = 1,Hopper-v4,4000,avg_return,0.0,13.854812
1,UTD = 1,Hopper-v4,4000,avg_return,50000.0,406.395977
2,UTD = 1,Hopper-v4,4000,avg_return,100000.0,697.883268
3,UTD = 1,Hopper-v4,4000,avg_return,150000.0,1536.789357
4,UTD = 1,Hopper-v4,4000,avg_return,200000.0,3178.249772
...,...,...,...,...,...,...
38954,UTD = 8,h1-stand-v0,0,avg_success,600000.0,0.000000
38955,UTD = 8,h1-stand-v0,0,avg_success,700000.0,0.000000
38956,UTD = 8,h1-stand-v0,0,avg_success,800000.0,0.000000
38957,UTD = 8,h1-stand-v0,0,avg_success,900000.0,0.000000


In [5]:
exp_names = eval_df['exp_name'].unique()
exp_names

array(['UTD = 1', 'UTD = 2', 'UTD = 4', 'UTD = 8'], dtype=object)

In [6]:
env_names = eval_df['env_name'].unique()
env_names

array(['Hopper-v4', 'Humanoid-v4', 'Ant-v4', 'Walker2d-v4',
       'HalfCheetah-v4', 'myo-pen-twirl-hard', 'myo-pen-twirl',
       'myo-key-turn-hard', 'myo-key-turn', 'myo-obj-hold-hard',
       'myo-obj-hold', 'myo-pose-hard', 'myo-pose', 'myo-reach-hard',
       'myo-reach', 'walker-run', 'walker-walk', 'walker-stand',
       'reacher-hard', 'reacher-easy', 'quadruped-run', 'quadruped-walk',
       'pendulum-swingup', 'hopper-stand', 'hopper-hop', 'fish-swim',
       'finger-turn-hard', 'finger-turn-easy', 'finger-spin',
       'cheetah-run', 'cartpole-swingup-sparse', 'cartpole-swingup',
       'cartpole-balance-sparse', 'cartpole-balance', 'ball-in-cup-catch',
       'acrobot-swingup', 'h1-sit-hard-v0', 'h1-walk-v0', 'h1-stair-v0',
       'h1-run-v0', 'h1-balance-simple-v0', 'dog-trot', 'dog-run',
       'dog-walk', 'dog-stand', 'humanoid-run', 'humanoid-walk',
       'humanoid-stand', 'h1-pole-v0', 'h1-slide-v0',
       'h1-balance-hard-v0', 'h1-sit-simple-v0', 'h1-maze-v0',
    

### Visualization

In [7]:
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_iqm(x),
  rly_metrics.aggregate_median(x),
  rly_metrics.aggregate_mean(x)])

In [10]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE
from scale_rl.envs.dmc import DMC_EASY_MEDIUM, DMC_HARD
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_RANDOM_SCORE, HB_SUCCESS_SCORE
from scale_rl.envs.myosuite import MYOSUITE_TASKS

def replace_hypen_to_underbar(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_hypen_to_underbar_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

MUJOCO_ALL = replace_hypen_to_underbar(MUJOCO_ALL)
DMC_EASY_MEDIUM = replace_hypen_to_underbar(DMC_EASY_MEDIUM)
DMC_HARD = replace_hypen_to_underbar(DMC_HARD)
HB_LOCOMOTION_NOHAND = replace_hypen_to_underbar(HB_LOCOMOTION_NOHAND)
MYOSUITE_TASKS = replace_hypen_to_underbar(MYOSUITE_TASKS)

HB_SUCCESS_SCORE = replace_hypen_to_underbar_dict(HB_SUCCESS_SCORE)
HB_RANDOM_SCORE = replace_hypen_to_underbar_dict(HB_RANDOM_SCORE)

DMC_ALL = [*DMC_EASY_MEDIUM, *DMC_HARD]

In [11]:
MUJ_STEPS = 1000000 # 1M
DMC_STEPS = 1000000 # 1M
MYO_STEPS = 1000000 # 1M
HB_STEPS = 1000000 # 1M

In [12]:
def full_results_per_env(
    eval_df,
    total_env_steps,
    metric_matrix_dict,
    aggregate_func,
    metric: str = "avg_return",
    columns: List[str] = ["UTD = 1", "UTD = 2", "UTD = 4", "UTD = 8"],
):
    experiments = eval_df["exp_name"].unique()
    assert sorted(columns) == sorted(experiments)
    environments = eval_df["env_name"].unique()
    
    aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
        metric_matrix_dict, aggregate_func, reps=10000
    )
    
    eval_df = eval_df[eval_df["metric"] == metric]
    eval_df = eval_df[eval_df["env_step"] <= total_env_steps]
    
    full_results_df = pd.DataFrame(columns=["Task"].append(columns))
    
    for _exp_name in experiments:
        _exp_data = eval_df[eval_df["exp_name"] == _exp_name]
        _num_seeds = _exp_data["seed"].nunique()
        print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
        
    for i, env in enumerate(environments):
        env_data = eval_df[eval_df["env_name"] == env]
        data = {"\\textbf{Task}": "\\texttt{" + str(env) + "}"}
        for j, exp in enumerate(columns):
            exp_data = env_data[env_data["exp_name"] == exp]
            if len(exp_data) == 0:
                continue
            # assert max(exp_data["env_step"]) == total_env_steps
            exp_data = exp_data[exp_data["env_step"] == max(exp_data["env_step"])]
            
            grouped_data = exp_data.groupby("env_step")["value"]

            env_steps = grouped_data.mean().index.values
            mean = float(grouped_data.mean().values)
            std_err = float(grouped_data.sem().values)
            low_CI = mean - 1.960 * std_err
            high_CI = mean + 1.960 * std_err
            
            if metric == "avg_success": 
                mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
            else:    
                mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
            
            
            data["\\textbf{" + f"{exp}" + "}"] = f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}"

        full_results_df = pd.concat([full_results_df, pd.DataFrame.from_dict(
                                data=[data], orient='columns')], ignore_index=True)

    
    for i, agg in enumerate(["IQM", "Median", "Mean"]):
        data = {"\\textbf{Task}": agg}
        for j, exp in enumerate(columns):
            mean = aggregate_scores[exp][i] 
            low_CI = aggregate_score_cis[exp][0][i] 
            high_CI = aggregate_score_cis[exp][1][i]
            
            if metric == "avg_success": 
                mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
            else:
                mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
            data["\\textbf{" + f"{exp}" + "}"] = f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}"
        
        full_results_df = pd.concat([full_results_df, pd.DataFrame.from_dict(
                            data=[data], orient='columns')], ignore_index=True)

    return full_results_df
        

#### MuJoCo (Simba 1M) 

In [13]:
mujoco_df = eval_df[eval_df['env_name'].isin(MUJOCO_ALL)].sort_values("env_name")
total_env_steps = MUJ_STEPS
metric = "avg_return"
mujoco_metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=mujoco_df,
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=MUJ_STEPS,
    metric_type="avg_return",
)

In [14]:
mujoco_full_results_df = full_results_per_env(
    mujoco_df,
    metric_matrix_dict=mujoco_metric_matrix_dict,
    total_env_steps=MUJ_STEPS,
    aggregate_func=aggregate_func,
    metric="avg_return",
    columns=["UTD = 1", "UTD = 2", "UTD = 4", "UTD = 8"],
)
mujoco_full_results_df

exp_name: UTD = 2 - num_seeds: 10
exp_name: UTD = 4 - num_seeds: 5
exp_name: UTD = 8 - num_seeds: 5
exp_name: UTD = 1 - num_seeds: 5


,\textbf{Task},\textbf{UTD = 1},\textbf{UTD = 2},\textbf{UTD = 4},\textbf{UTD = 8}
0,\texttt{Ant-v4},"7405 \textcolor{gray}{[7315, 7496]}","7429 \textcolor{gray}{[7209, 7649]}","7230 \textcolor{gray}{[6968, 7492]}","6940 \textcolor{gray}{[6431, 7449]}"
1,\texttt{HalfCheetah-v4},"11425 \textcolor{gray}{[10798, 12052]}","12022 \textcolor{gray}{[11640, 12404]}","12007 \textcolor{gray}{[11458, 12557]}","11592 \textcolor{gray}{[9956, 13229]}"
2,\texttt{Hopper-v4},"3579 \textcolor{gray}{[3311, 3847]}","4053 \textcolor{gray}{[3928, 4178]}","4003 \textcolor{gray}{[3647, 4359]}","4151 \textcolor{gray}{[4033, 4269]}"
3,\texttt{Humanoid-v4},"7696 \textcolor{gray}{[4385, 11008]}","10545 \textcolor{gray}{[10195, 10896]}","11133 \textcolor{gray}{[10908, 11358]}","11703 \textcolor{gray}{[11282, 12125]}"
4,\texttt{Walker2d-v4},"6069 \textcolor{gray}{[5724, 6414]}","6938 \textcolor{gray}{[6691, 7185]}","6804 \textcolor{gray}{[6459, 7148]}","6163 \textcolor{gray}{[4522, 7804]}"
5,IQM,"1.433 \textcolor{gray}{[1.225, 1.648]}","1.637 \textcolor{gray}{[1.471, 1.788]}","1.617 \textcolor{gray}{[1.402, 1.83]}","1.581 \textcolor{gray}{[1.358, 1.82]}"
6,Median,"1.468 \textcolor{gray}{[1.269, 1.625]}","1.616 \textcolor{gray}{[1.491, 1.743]}","1.615 \textcolor{gray}{[1.438, 1.809]}","1.602 \textcolor{gray}{[1.377, 1.821]}"
7,Mean,"1.418 \textcolor{gray}{[1.264, 1.569]}","1.617 \textcolor{gray}{[1.513, 1.719]}","1.62 \textcolor{gray}{[1.47, 1.773]}","1.598 \textcolor{gray}{[1.419, 1.78]}"


In [15]:
mujoco_latex_df = mujoco_full_results_df.to_latex(index=False)
mujoco_latex_df

'\\begin{tabular}{lllll}\n\\toprule\n\\textbf{Task} & \\textbf{UTD = 1} & \\textbf{UTD = 2} & \\textbf{UTD = 4} & \\textbf{UTD = 8} \\\\\n\\midrule\n\\texttt{Ant-v4} & 7405 \\textcolor{gray}{[7315, 7496]} & 7429 \\textcolor{gray}{[7209, 7649]} & 7230 \\textcolor{gray}{[6968, 7492]} & 6940 \\textcolor{gray}{[6431, 7449]} \\\\\n\\texttt{HalfCheetah-v4} & 11425 \\textcolor{gray}{[10798, 12052]} & 12022 \\textcolor{gray}{[11640, 12404]} & 12007 \\textcolor{gray}{[11458, 12557]} & 11592 \\textcolor{gray}{[9956, 13229]} \\\\\n\\texttt{Hopper-v4} & 3579 \\textcolor{gray}{[3311, 3847]} & 4053 \\textcolor{gray}{[3928, 4178]} & 4003 \\textcolor{gray}{[3647, 4359]} & 4151 \\textcolor{gray}{[4033, 4269]} \\\\\n\\texttt{Humanoid-v4} & 7696 \\textcolor{gray}{[4385, 11008]} & 10545 \\textcolor{gray}{[10195, 10896]} & 11133 \\textcolor{gray}{[10908, 11358]} & 11703 \\textcolor{gray}{[11282, 12125]} \\\\\n\\texttt{Walker2d-v4} & 6069 \\textcolor{gray}{[5724, 6414]} & 6938 \\textcolor{gray}{[6691, 7185]

In [16]:
with open('utd_mujoco.tex', 'w') as tf:
    tf.write(mujoco_latex_df)

#### DMC EM

In [17]:
dmc_easy_eval_df = eval_df[eval_df['env_name'].isin(DMC_EASY_MEDIUM)]
dmc_easy_eval_df = dmc_easy_eval_df.sort_values(by = ['env_name'])

dmc_easy_eval_df["value"] = dmc_easy_eval_df["value"] / 1000
dmc_easy_metric_matrix_dict = generate_metric_matrix_dict(
    dmc_easy_eval_df, 
    env_step=DMC_STEPS, 
    metric_type="avg_return",
)
dmc_easy_eval_df["value"] = dmc_easy_eval_df["value"] * 1000
dmc_easy_full_results_df = full_results_per_env(
    dmc_easy_eval_df,
    metric_matrix_dict=dmc_easy_metric_matrix_dict,
    total_env_steps=DMC_STEPS,
    aggregate_func=aggregate_func,
    columns=["UTD = 1", "UTD = 2", "UTD = 4", "UTD = 8"],
)
dmc_easy_full_results_df

exp_name: UTD = 8 - num_seeds: 5
exp_name: UTD = 1 - num_seeds: 5
exp_name: UTD = 4 - num_seeds: 5
exp_name: UTD = 2 - num_seeds: 10


,\textbf{Task},\textbf{UTD = 1},\textbf{UTD = 2},\textbf{UTD = 4},\textbf{UTD = 8}
0,\texttt{acrobot-swingup},"413 \textcolor{gray}{[376, 450]}","436 \textcolor{gray}{[391, 482]}","458 \textcolor{gray}{[359, 558]}","477 \textcolor{gray}{[438, 516]}"
1,\texttt{ball-in-cup-catch},"981 \textcolor{gray}{[977, 985]}","982 \textcolor{gray}{[980, 984]}","982 \textcolor{gray}{[979, 986]}","982 \textcolor{gray}{[979, 985]}"
2,\texttt{cartpole-balance},"999 \textcolor{gray}{[999, 999]}","999 \textcolor{gray}{[999, 999]}","999 \textcolor{gray}{[999, 999]}","999 \textcolor{gray}{[999, 999]}"
3,\texttt{cartpole-balance-sparse},"1000 \textcolor{gray}{[1000, 1000]}","967 \textcolor{gray}{[904, 1030]}","1000 \textcolor{gray}{[1000, 1000]}","1000 \textcolor{gray}{[1000, 1000]}"
4,\texttt{cartpole-swingup},"881 \textcolor{gray}{[881, 881]}","880 \textcolor{gray}{[876, 883]}","880 \textcolor{gray}{[879, 881]}","881 \textcolor{gray}{[880, 882]}"
5,\texttt{cartpole-swingup-sparse},"845 \textcolor{gray}{[843, 848]}","848 \textcolor{gray}{[848, 849]}","848 \textcolor{gray}{[848, 849]}","841 \textcolor{gray}{[824, 858]}"
6,\texttt{cheetah-run},"917 \textcolor{gray}{[913, 920]}","920 \textcolor{gray}{[918, 922]}","902 \textcolor{gray}{[868, 937]}","916 \textcolor{gray}{[912, 920]}"
7,\texttt{finger-spin},"940 \textcolor{gray}{[895, 985]}","891 \textcolor{gray}{[810, 972]}","762 \textcolor{gray}{[608, 915]}","910 \textcolor{gray}{[790, 1030]}"
8,\texttt{finger-turn-easy},"951 \textcolor{gray}{[916, 987]}","953 \textcolor{gray}{[925, 980]}","954 \textcolor{gray}{[917, 992]}","936 \textcolor{gray}{[857, 1014]}"
9,\texttt{finger-turn-hard},"928 \textcolor{gray}{[885, 972]}","951 \textcolor{gray}{[925, 977]}","902 \textcolor{gray}{[866, 939]}","950 \textcolor{gray}{[910, 990]}"


In [18]:
dmc_easy_latex_df = dmc_easy_full_results_df.to_latex(index=False)
dmc_easy_latex_df

'\\begin{tabular}{lllll}\n\\toprule\n\\textbf{Task} & \\textbf{UTD = 1} & \\textbf{UTD = 2} & \\textbf{UTD = 4} & \\textbf{UTD = 8} \\\\\n\\midrule\n\\texttt{acrobot-swingup} & 413 \\textcolor{gray}{[376, 450]} & 436 \\textcolor{gray}{[391, 482]} & 458 \\textcolor{gray}{[359, 558]} & 477 \\textcolor{gray}{[438, 516]} \\\\\n\\texttt{ball-in-cup-catch} & 981 \\textcolor{gray}{[977, 985]} & 982 \\textcolor{gray}{[980, 984]} & 982 \\textcolor{gray}{[979, 986]} & 982 \\textcolor{gray}{[979, 985]} \\\\\n\\texttt{cartpole-balance} & 999 \\textcolor{gray}{[999, 999]} & 999 \\textcolor{gray}{[999, 999]} & 999 \\textcolor{gray}{[999, 999]} & 999 \\textcolor{gray}{[999, 999]} \\\\\n\\texttt{cartpole-balance-sparse} & 1000 \\textcolor{gray}{[1000, 1000]} & 967 \\textcolor{gray}{[904, 1030]} & 1000 \\textcolor{gray}{[1000, 1000]} & 1000 \\textcolor{gray}{[1000, 1000]} \\\\\n\\texttt{cartpole-swingup} & 881 \\textcolor{gray}{[881, 881]} & 880 \\textcolor{gray}{[876, 883]} & 880 \\textcolor{gray}{[87

In [19]:
with open('utd_dmc_easy.tex', 'w') as tf:
    tf.write(dmc_easy_latex_df)

#### DMC Hard

In [20]:
dmc_hard_eval_df = eval_df[eval_df['env_name'].isin(DMC_HARD)]
dmc_hard_eval_df = dmc_hard_eval_df.sort_values(by = ['env_name'])

dmc_hard_eval_df["value"] = dmc_hard_eval_df["value"] / 1000
dmc_hard_metric_matrix_dict = generate_metric_matrix_dict(
    dmc_hard_eval_df, 
    env_step=DMC_STEPS, 
    metric_type="avg_return",
)
dmc_hard_eval_df["value"] = dmc_hard_eval_df["value"] * 1000
dmc_hard_full_results_df = full_results_per_env(
    dmc_hard_eval_df,
    metric_matrix_dict=dmc_hard_metric_matrix_dict,
    total_env_steps=DMC_STEPS,
    aggregate_func=aggregate_func,
    columns=["UTD = 1", "UTD = 2", "UTD = 4", "UTD = 8"],
)
dmc_hard_full_results_df

exp_name: UTD = 4 - num_seeds: 5
exp_name: UTD = 2 - num_seeds: 10
exp_name: UTD = 8 - num_seeds: 5
exp_name: UTD = 1 - num_seeds: 5


,\textbf{Task},\textbf{UTD = 1},\textbf{UTD = 2},\textbf{UTD = 4},\textbf{UTD = 8}
0,\texttt{dog-run},"477 \textcolor{gray}{[429, 525]}","562 \textcolor{gray}{[516, 608]}","655 \textcolor{gray}{[620, 691]}","555 \textcolor{gray}{[523, 587]}"
1,\texttt{dog-stand},"967 \textcolor{gray}{[959, 974]}","981 \textcolor{gray}{[977, 985]}","967 \textcolor{gray}{[960, 974]}","972 \textcolor{gray}{[967, 976]}"
2,\texttt{dog-trot},"850 \textcolor{gray}{[810, 890]}","861 \textcolor{gray}{[772, 950]}","846 \textcolor{gray}{[782, 910]}","898 \textcolor{gray}{[888, 909]}"
3,\texttt{dog-walk},"921 \textcolor{gray}{[912, 930]}","935 \textcolor{gray}{[927, 944]}","923 \textcolor{gray}{[905, 941]}","949 \textcolor{gray}{[945, 953]}"
4,\texttt{humanoid-run},"183 \textcolor{gray}{[164, 203]}","194 \textcolor{gray}{[182, 207]}","272 \textcolor{gray}{[230, 313]}","253 \textcolor{gray}{[228, 278]}"
5,\texttt{humanoid-stand},"660 \textcolor{gray}{[585, 734]}","916 \textcolor{gray}{[886, 945]}","928 \textcolor{gray}{[926, 930]}","933 \textcolor{gray}{[924, 941]}"
6,\texttt{humanoid-walk},"568 \textcolor{gray}{[533, 603]}","651 \textcolor{gray}{[590, 713]}","818 \textcolor{gray}{[751, 885]}","819 \textcolor{gray}{[762, 877]}"
7,IQM,"0.713 \textcolor{gray}{[0.598, 0.809]}","0.808 \textcolor{gray}{[0.725, 0.88]}","0.851 \textcolor{gray}{[0.755, 0.916]}","0.849 \textcolor{gray}{[0.727, 0.924]}"
8,Median,"0.666 \textcolor{gray}{[0.563, 0.774]}","0.729 \textcolor{gray}{[0.655, 0.81]}","0.771 \textcolor{gray}{[0.678, 0.868]}","0.767 \textcolor{gray}{[0.652, 0.861]}"
9,Mean,"0.669 \textcolor{gray}{[0.581, 0.753]}","0.729 \textcolor{gray}{[0.663, 0.791]}","0.769 \textcolor{gray}{[0.687, 0.845]}","0.759 \textcolor{gray}{[0.67, 0.84]}"


In [21]:
dmc_hard_latex_df = dmc_hard_full_results_df.to_latex(index=False)
dmc_hard_latex_df

'\\begin{tabular}{lllll}\n\\toprule\n\\textbf{Task} & \\textbf{UTD = 1} & \\textbf{UTD = 2} & \\textbf{UTD = 4} & \\textbf{UTD = 8} \\\\\n\\midrule\n\\texttt{dog-run} & 477 \\textcolor{gray}{[429, 525]} & 562 \\textcolor{gray}{[516, 608]} & 655 \\textcolor{gray}{[620, 691]} & 555 \\textcolor{gray}{[523, 587]} \\\\\n\\texttt{dog-stand} & 967 \\textcolor{gray}{[959, 974]} & 981 \\textcolor{gray}{[977, 985]} & 967 \\textcolor{gray}{[960, 974]} & 972 \\textcolor{gray}{[967, 976]} \\\\\n\\texttt{dog-trot} & 850 \\textcolor{gray}{[810, 890]} & 861 \\textcolor{gray}{[772, 950]} & 846 \\textcolor{gray}{[782, 910]} & 898 \\textcolor{gray}{[888, 909]} \\\\\n\\texttt{dog-walk} & 921 \\textcolor{gray}{[912, 930]} & 935 \\textcolor{gray}{[927, 944]} & 923 \\textcolor{gray}{[905, 941]} & 949 \\textcolor{gray}{[945, 953]} \\\\\n\\texttt{humanoid-run} & 183 \\textcolor{gray}{[164, 203]} & 194 \\textcolor{gray}{[182, 207]} & 272 \\textcolor{gray}{[230, 313]} & 253 \\textcolor{gray}{[228, 278]} \\\\\n\\

In [22]:
with open('utd_dmc_hard.tex', 'w') as tf:
    tf.write(dmc_hard_latex_df)

#### MyoSuite

In [23]:
myosuite_eval_df = eval_df[eval_df['env_name'].isin(MYOSUITE_TASKS)]
myosuite_metric_matrix_dict = generate_metric_matrix_dict(
    myosuite_eval_df, 
    env_step=MYO_STEPS, 
    metric_type="avg_success",
)

myosuite_full_results_df = full_results_per_env(
    myosuite_eval_df,
    metric_matrix_dict=myosuite_metric_matrix_dict,
    total_env_steps=MYO_STEPS,
    aggregate_func=aggregate_func,
    metric="avg_success",
    columns=["UTD = 1", "UTD = 2", "UTD = 4", "UTD = 8"],
)
myosuite_full_results_df

exp_name: UTD = 1 - num_seeds: 5
exp_name: UTD = 2 - num_seeds: 10
exp_name: UTD = 4 - num_seeds: 5
exp_name: UTD = 8 - num_seeds: 5


,\textbf{Task},\textbf{UTD = 1},\textbf{UTD = 2},\textbf{UTD = 4},\textbf{UTD = 8}
0,\texttt{myo-pen-twirl-hard},"76.0 \textcolor{gray}{[57.8, 94.2]}","93.0 \textcolor{gray}{[88.8, 97.2]}","92.0 \textcolor{gray}{[84.7, 99.3]}","98.0 \textcolor{gray}{[94.1, 101.9]}"
1,\texttt{myo-pen-twirl},"100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
2,\texttt{myo-key-turn-hard},"46.0 \textcolor{gray}{[8.5, 83.5]}","62.0 \textcolor{gray}{[42.7, 81.3]}","70.0 \textcolor{gray}{[34.9, 105.1]}","80.0 \textcolor{gray}{[49.6, 110.4]}"
3,\texttt{myo-key-turn},"80.0 \textcolor{gray}{[40.8, 119.2]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
4,\texttt{myo-obj-hold-hard},"100.0 \textcolor{gray}{[100.0, 100.0]}","98.0 \textcolor{gray}{[95.4, 100.6]}","98.0 \textcolor{gray}{[94.1, 101.9]}","92.0 \textcolor{gray}{[84.7, 99.3]}"
5,\texttt{myo-obj-hold},"100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
6,\texttt{myo-pose-hard},"0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}"
7,\texttt{myo-pose},"100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
8,\texttt{myo-reach-hard},"92.0 \textcolor{gray}{[84.7, 99.3]}","94.0 \textcolor{gray}{[87.3, 100.7]}","98.0 \textcolor{gray}{[94.1, 101.9]}","96.0 \textcolor{gray}{[91.2, 100.8]}"
9,\texttt{myo-reach},"100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"


In [24]:
myosuite_latex_df = myosuite_full_results_df.to_latex(index=False)
myosuite_latex_df

'\\begin{tabular}{lllll}\n\\toprule\n\\textbf{Task} & \\textbf{UTD = 1} & \\textbf{UTD = 2} & \\textbf{UTD = 4} & \\textbf{UTD = 8} \\\\\n\\midrule\n\\texttt{myo-pen-twirl-hard} & 76.0 \\textcolor{gray}{[57.8, 94.2]} & 93.0 \\textcolor{gray}{[88.8, 97.2]} & 92.0 \\textcolor{gray}{[84.7, 99.3]} & 98.0 \\textcolor{gray}{[94.1, 101.9]} \\\\\n\\texttt{myo-pen-twirl} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} \\\\\n\\texttt{myo-key-turn-hard} & 46.0 \\textcolor{gray}{[8.5, 83.5]} & 62.0 \\textcolor{gray}{[42.7, 81.3]} & 70.0 \\textcolor{gray}{[34.9, 105.1]} & 80.0 \\textcolor{gray}{[49.6, 110.4]} \\\\\n\\texttt{myo-key-turn} & 80.0 \\textcolor{gray}{[40.8, 119.2]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} \\\\\n\\texttt{myo-obj-hold-hard} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 98.

In [25]:
with open('utd_myosuite.tex', 'w') as tf:
    tf.write(myosuite_latex_df)

#### Humanoid Bench

In [26]:
hb_df = eval_df[eval_df['env_name'].isin(HB_LOCOMOTION_NOHAND)]
hb_metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=hb_df,
        random_score_dict=HB_RANDOM_SCORE,
        base_score_dict=HB_SUCCESS_SCORE,
    ),
    env_step=HB_STEPS,
    metric_type="avg_return",
)

hb_full_results_df = full_results_per_env(
    hb_df,
    metric_matrix_dict=hb_metric_matrix_dict,
    total_env_steps=HB_STEPS,
    aggregate_func=aggregate_func,
    metric="avg_return",
    columns=["UTD = 1", "UTD = 2", "UTD = 4", "UTD = 8"],
)
hb_full_results_df

exp_name: UTD = 1 - num_seeds: 5
exp_name: UTD = 2 - num_seeds: 10
exp_name: UTD = 4 - num_seeds: 5
exp_name: UTD = 8 - num_seeds: 5


,\textbf{Task},\textbf{UTD = 1},\textbf{UTD = 2},\textbf{UTD = 4},\textbf{UTD = 8}
0,\texttt{h1-sit-hard-v0},"681 \textcolor{gray}{[506, 857]}","679 \textcolor{gray}{[548, 811]}","719 \textcolor{gray}{[664, 773]}","810 \textcolor{gray}{[784, 836]}"
1,\texttt{h1-walk-v0},"732 \textcolor{gray}{[522, 941]}","845 \textcolor{gray}{[840, 850]}","846 \textcolor{gray}{[841, 851]}","844 \textcolor{gray}{[840, 847]}"
2,\texttt{h1-stair-v0},"473 \textcolor{gray}{[444, 503]}","493 \textcolor{gray}{[467, 518]}","546 \textcolor{gray}{[541, 550]}","532 \textcolor{gray}{[512, 552]}"
3,\texttt{h1-run-v0},"247 \textcolor{gray}{[152, 342]}","415 \textcolor{gray}{[307, 524]}","318 \textcolor{gray}{[176, 461]}","425 \textcolor{gray}{[293, 558]}"
4,\texttt{h1-balance-simple-v0},"806 \textcolor{gray}{[773, 839]}","723 \textcolor{gray}{[651, 795]}","775 \textcolor{gray}{[719, 831]}","813 \textcolor{gray}{[797, 828]}"
5,\texttt{h1-pole-v0},"769 \textcolor{gray}{[758, 780]}","791 \textcolor{gray}{[785, 797]}","799 \textcolor{gray}{[780, 817]}","827 \textcolor{gray}{[787, 868]}"
6,\texttt{h1-slide-v0},"412 \textcolor{gray}{[279, 544]}","487 \textcolor{gray}{[404, 571]}","544 \textcolor{gray}{[500, 588]}","534 \textcolor{gray}{[505, 563]}"
7,\texttt{h1-balance-hard-v0},"135 \textcolor{gray}{[111, 160]}","143 \textcolor{gray}{[128, 157]}","128 \textcolor{gray}{[118, 139]}","167 \textcolor{gray}{[157, 178]}"
8,\texttt{h1-sit-simple-v0},"873 \textcolor{gray}{[868, 879]}","875 \textcolor{gray}{[870, 880]}","908 \textcolor{gray}{[861, 955]}","867 \textcolor{gray}{[839, 894]}"
9,\texttt{h1-maze-v0},"350 \textcolor{gray}{[332, 368]}","313 \textcolor{gray}{[287, 340]}","343 \textcolor{gray}{[327, 359]}","338 \textcolor{gray}{[325, 351]}"


In [27]:
hb_latex_df = hb_full_results_df.to_latex(index=False)
hb_latex_df

'\\begin{tabular}{lllll}\n\\toprule\n\\textbf{Task} & \\textbf{UTD = 1} & \\textbf{UTD = 2} & \\textbf{UTD = 4} & \\textbf{UTD = 8} \\\\\n\\midrule\n\\texttt{h1-sit-hard-v0} & 681 \\textcolor{gray}{[506, 857]} & 679 \\textcolor{gray}{[548, 811]} & 719 \\textcolor{gray}{[664, 773]} & 810 \\textcolor{gray}{[784, 836]} \\\\\n\\texttt{h1-walk-v0} & 732 \\textcolor{gray}{[522, 941]} & 845 \\textcolor{gray}{[840, 850]} & 846 \\textcolor{gray}{[841, 851]} & 844 \\textcolor{gray}{[840, 847]} \\\\\n\\texttt{h1-stair-v0} & 473 \\textcolor{gray}{[444, 503]} & 493 \\textcolor{gray}{[467, 518]} & 546 \\textcolor{gray}{[541, 550]} & 532 \\textcolor{gray}{[512, 552]} \\\\\n\\texttt{h1-run-v0} & 247 \\textcolor{gray}{[152, 342]} & 415 \\textcolor{gray}{[307, 524]} & 318 \\textcolor{gray}{[176, 461]} & 425 \\textcolor{gray}{[293, 558]} \\\\\n\\texttt{h1-balance-simple-v0} & 806 \\textcolor{gray}{[773, 839]} & 723 \\textcolor{gray}{[651, 795]} & 775 \\textcolor{gray}{[719, 831]} & 813 \\textcolor{gray}{

In [28]:
with open('utd_hbench.tex', 'w') as tf:
    tf.write(hb_latex_df)